# 11 — Dynamic Routing

**Learning objective:** dispatch a runtime-dependent number of work items while enforcing maximum fan-out, depth, and allowed-tool policy.

Static topology means known nodes and transitions. Dynamic execution means one run may schedule a different number of instances or routes. Dynamic does not mean arbitrary or uncontrolled.

## Mental model and topology

```mermaid
flowchart LR
    accTitle: Bounded dynamic map reduce
    accDescr: Planning caps discovered work, dispatch creates a bounded number of worker executions, and a reducer merges results before aggregation.

    discover[Discovered items] --> plan[Plan within bounds]
    plan --> dispatch{Dynamic dispatch}
    dispatch --> worker_a[Worker instance]
    dispatch --> worker_b[Worker instance]
    dispatch --> worker_n[Up to maximum workers]
    worker_a --> aggregate[Aggregate]
    worker_b --> aggregate
    worker_n --> aggregate
    aggregate --> end_node([End])
```

## State, reducer, and control policy

Planning deduplicates discovered items and stores at most `MAX_WORKERS`. `Send` creates one worker input per planned item. Each worker can select only from `ALLOWED_TOOLS`; an append reducer retains every result. At `MAX_DEPTH`, dispatch creates no new workers. Production policy should additionally cap attempts, cost/token budget, and wall-clock timeout.

In [1]:
from graph_engineering.dynamic import (
    ALLOWED_TOOLS, MAX_DEPTH, MAX_WORKERS, build_dynamic_graph,
)

graph = build_dynamic_graph()
print("Policy:", {"max_workers": MAX_WORKERS, "max_depth": MAX_DEPTH, "allowed_tools": ALLOWED_TOOLS})

Policy: {'max_workers': 5, 'max_depth': 3, 'allowed_tools': ('search', 'catalog')}


In [2]:
discovered = [f"source-{index}" for index in range(8)]
result = graph.invoke(
    {"discovered_items": discovered, "depth": 0, "allowed_tools": ALLOWED_TOOLS}
)
print("Discovered:", len(discovered))
print("Workers dispatched:", result["fan_out_count"])
print("Merged results:", len(result["worker_results"]))
print("Items processed:", sorted(item for item, _, _ in result["worker_results"]))

Discovered: 8
Workers dispatched: 5
Merged results: 5
Items processed: ['source-0', 'source-1', 'source-2', 'source-3', 'source-4']


## Failure considerations

Unbounded discovery can create a fan-out explosion, cost spike, rate-limit storm, or recursive loop. A dynamic graph must define maximum workers, maximum depth, maximum attempts, cost budget, timeout, allowed tools, allowed routes, cancellation behavior, and partial-failure aggregation. Hard limits belong in code.

In [3]:
blocked = graph.invoke({"discovered_items": ["a", "b"], "depth": MAX_DEPTH})
print("Workers at depth limit:", blocked["fan_out_count"])
print("Termination:", blocked["termination_reason"])

Workers at depth limit: 0
Termination: depth_limit_reached


## What to modify

Add duplicate discoveries, request a disallowed tool, or change the worker bound and verify the reducer and trace. Then define a partial-failure rule: all, quorum, or any success.

**Next:** [Evaluating Graph Systems](../docs/10_evaluating_graph_systems.md) measures the whole system rather than only its final answer.